# Averis spam dataset audit

This notebook audits only the 520 labeled Averis emails. It does not use the UCI SMS corpus or any other external training data.

In [1]:
import json
from collections import Counter
from pathlib import Path

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists())
DATA_ROOT = ROOT / "docs-provided/problem-statement/sdoc-hackathon-docker/data_v2"
INBOX = DATA_ROOT / "inbox"
labels = json.loads((DATA_ROOT / "ground_truth.json").read_text())

rows = []
for path in sorted(INBOX.glob("*.json")):
    email = json.loads(path.read_text())
    rows.append(
        {
            "email_id": email["email_id"],
            "sender": email.get("from", ""),
            "subject": email.get("subject", ""),
            "body": email.get("body", ""),
            "is_spam": labels[email["email_id"]]["category"] == "SPAM",
        }
    )

len(rows), Counter(row["is_spam"] for row in rows)

(520, Counter({False: 480, True: 40}))

In [2]:
def model_text(row):
    return f"From: {row['sender']}\nSubject: {row['subject']}\nBody: {row['body']}"

summary = {
    "records": len(rows),
    "spam": sum(row["is_spam"] for row in rows),
    "ham": sum(not row["is_spam"] for row in rows),
    "spam_rate": sum(row["is_spam"] for row in rows) / len(rows),
    "unique_full_text": len({model_text(row) for row in rows}),
    "unique_subjects": len({row["subject"] for row in rows}),
    "unique_bodies": len({row["body"] for row in rows}),
    "unique_spam_subjects": len({row["subject"] for row in rows if row["is_spam"]}),
    "unique_spam_bodies": len({row["body"] for row in rows if row["is_spam"]}),
}
print(json.dumps(summary, indent=2))

{
  "records": 520,
  "spam": 40,
  "ham": 480,
  "spam_rate": 0.07692307692307693,
  "unique_full_text": 519,
  "unique_subjects": 474,
  "unique_bodies": 449,
  "unique_spam_subjects": 9,
  "unique_spam_bodies": 6
}


In [3]:
duplicates = {}
for row in rows:
    duplicates.setdefault(model_text(row), []).append(row["email_id"])
print([ids for ids in duplicates.values() if len(ids) > 1])

[['email_206', 'email_450']]


## Limitations

- Only 40 of 520 records are spam, so spam metrics have high uncertainty.
- The dataset is synthetic and template-generated: spam has only nine unique subjects and six unique bodies.
- A random stratified holdout measures performance on this generator, not real-world email drift.
- Exact full-text duplicates are removed before model selection to prevent direct train/test leakage.
- Results must not be generalized beyond this dataset without evaluation on independently collected email.